# 04 — GRU (Gated Recurrent Unit)
Lighter alternative to LSTM for sequential customer data.

Run `01_eda_preprocessing.ipynb` first.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.sequence import pad_sequences

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)

In [ ]:
df = pd.read_csv('data/cleaned.csv')
feature_cols = [c for c in df.columns if c not in ['Customer_ID','Credit_Score_Enc']]
sequences, targets = [], []
for _, group in df.groupby('Customer_ID'):
    sort_col = 'Month_ordinal' if 'Month_ordinal' in group.columns else feature_cols[0]
    sequences.append(group.sort_values(sort_col)[feature_cols].values)
    targets.append(group['Credit_Score_Enc'].mode()[0])
MAX_LEN = max(len(s) for s in sequences)
X_seq = pad_sequences(sequences, maxlen=MAX_LEN, dtype='float32', padding='post', value=0.0)
y_seq = np.array(targets)
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X_seq, y_seq, test_size=0.30, stratify=y_seq, random_state=SEED)
X_vl, X_ts, y_vl, y_ts = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)
cw = compute_class_weight('balanced', classes=np.unique(y_tr), y=y_tr)
class_weights = dict(enumerate(cw))

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(X_tr.shape[1], X_tr.shape[2])),
    layers.Masking(mask_value=0.0),
    layers.GRU(64, return_sequences=True),
    layers.Dropout(0.3),
    layers.BatchNormalization(),
    layers.GRU(32, return_sequences=True),
    layers.Dropout(0.3),
    layers.BatchNormalization(),
    layers.GRU(32),
    layers.Dropout(0.3),
    layers.BatchNormalization(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.BatchNormalization(),
    layers.Dense(3, activation='softmax')
], name='GRU')
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

history = model.fit(X_tr, y_tr, validation_data=(X_vl, y_vl),
                    epochs=20, batch_size=32, class_weight=class_weights, verbose=1)

In [ ]:
y_pred = np.argmax(model.predict(X_ts), axis=1)
print(f'GRU Test Accuracy: {accuracy_score(y_ts, y_pred):.4f}')
print(classification_report(y_ts, y_pred, target_names=['Poor','Standard','Good']))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['accuracy'], label='Train'); axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('GRU Accuracy Over Epochs'); axes[0].legend()
cm = confusion_matrix(y_ts, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Poor','Standard','Good'], yticklabels=['Poor','Standard','Good'])
axes[1].set_title('GRU Confusion Matrix')
plt.tight_layout(); plt.savefig('results/gru_results.png', dpi=150); plt.show()